# ChatPromptTemplate的高级特性

## 1、部分变量预填充partial()
预填充某些固定不变的变量，创建模板的变体。
使用场景：
- 某些变量在所有调用中都相同
- 需要为不同用户/场景创建定制模板

举例

In [1]:
from langchain_core.prompts import ChatPromptTemplate
# 原始模板
template = ChatPromptTemplate.from_messages([
("system", "你是{role}，目标用户是{audience}"),
("user", "{task}")
])
# 部分填充
customer_support_template = template.partial(
role="客服专员",
audience="普通用户"
)
# 现在只需要提供 task
messages = customer_support_template.invoke({"task":"解释退款政策"})
print(messages)

messages=[SystemMessage(content='你是客服专员，目标用户是普通用户', additional_kwargs={}, response_metadata={}), HumanMessage(content='解释退款政策', additional_kwargs={}, response_metadata={})]


举例2：

In [2]:
# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
("system", "你是{department}的{role}"),
("user", "{task}")
])
# IT 部门
it_template = base_template.partial(
department="IT 部门",
role="技术支持"
)
# 销售部门
sales_template = base_template.partial(
department="销售部门",
role="销售顾问"
)
sales_template.invoke({"task":"为什么每年年底汽车会促销"})

ChatPromptValue(messages=[SystemMessage(content='你是销售部门的销售顾问', additional_kwargs={}, response_metadata={}), HumanMessage(content='为什么每年年底汽车会促销', additional_kwargs={}, response_metadata={})])

# 2、消息占位符
当你不确定消息提示模板使用什么角色，或者希望在格式化过程中 插入消息列表 时，该怎么办？ 这就
需要使用消息占位符，负责在特定位置添加消息列表。
- 使用场景：多轮对话系统存储历史消息以及Agent的中间步骤处理此功能非常有用。

## 2.1 JSON格式
举例

In [3]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages(
[("system", "你是一个有用的AI助手"),
("placeholder", "{conversation}"),]
)
prompt_value = template.invoke(
{
    "conversation": [
    ("human", "你好!"),
    ("ai", "今天我能帮你做什么？"),
    ("human", "你能给我做一个冰激凌吗？"),
    ("ai", "抱歉，我没有这样的能力"),]
}
    )
print(prompt_value)

messages=[SystemMessage(content='你是一个有用的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好!', additional_kwargs={}, response_metadata={}), AIMessage(content='今天我能帮你做什么？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能给我做一个冰激凌吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='抱歉，我没有这样的能力', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 2.2 方式2：MessagesPlaceholder实例
举例1

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
prompt_template = ChatPromptTemplate.from_messages([
("system", "You are a helpful assistant"),
MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs": [HumanMessage(content="hi!")]})
#prompt_template.format_messages(msgs=[HumanMessage(content="hi!")])

举例2：存储对话历史内容

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt_template = ChatPromptTemplate.from_messages(
[
("system", "你是一个非常友好的AI助手"),
MessagesPlaceholder(variable_name="history"),
("human", "{question}")
]
)
prompt_template.invoke(
{
"history": [("human", "5 + 2 = ?"),
("ai", "5 + 2 = 7")
],
"question": "结果再乘以4呢？"
}
)

## 2.3 可复用的模板库

在实际项目中，建议创建模板库。


举例1：
templates.py文件声明如下

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
class PromptLibrary:
    """可复用的提示词模板库"""
TRANSLATOR = ChatPromptTemplate.from_messages([
("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
("user", "翻译以下文本：\n{text}")
])
CODE_REVIEWER = ChatPromptTemplate.from_messages([
("system", "你是{language}代码审查专家，重点关注{focus}"),
("user", "审查代码：\n```{language}\n{code}\n```")
])
SUMMARIZER = ChatPromptTemplate.from_messages([
("system", "你是内容摘要专家"),
("user", "将以下内容总结为{num}个要点：\n{content}")
])
TUTOR = ChatPromptTemplate.from_messages([
("system", "你是{subject}导师，学生水平：{level}"),
("user", "{question}")
])